# 🧠 XAI — Explicabilidad del Modelo

Interpretación del modelo entrenado usando:
- **Feature importances** (intrínsecas del modelo)
- **SHAP** (explicaciones agnósticas)

Objetivo: entender *por qué* el modelo toma las decisiones que toma.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib

from src.modules.iris_classifier.data_processing.loader import load_splits

%matplotlib inline

## 1. Cargar modelo y datos

In [ ]:
model = joblib.load("../data/models/artifacts/iris_classifier/model.pkl")
X_train, X_test, y_train, y_test = load_splits()

FEATURE_NAMES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET_NAMES = ["setosa", "versicolor", "virginica"]

## 2. Feature Importances (intrínsecas)

In [ ]:
# Solo disponible para modelos basados en árboles
if hasattr(model, "feature_importances_"):
    importances = pd.Series(model.feature_importances_, index=FEATURE_NAMES)
    importances = importances.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 4))
    importances.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("Feature Importances (modelo entrenado)")
    ax.set_xlabel("Importancia")
    plt.tight_layout()
    plt.show()
else:
    print("El modelo no tiene feature_importances_. Se usará solo SHAP.")

## 3. SHAP — Valores Shapley

SHAP asigna a cada feature una contribución a la predicción de cada muestra.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

## 4. SHAP — Summary Plot (global)

In [ ]:
# Summary plot para cada clase
for i, name in enumerate(TARGET_NAMES):
    print(f"\n--- Clase: {name} ---")
    shap.summary_plot(shap_values[i], X_test, feature_names=FEATURE_NAMES, show=True)

## 5. SHAP — Bar Plot (importancia media)

In [ ]:
shap.summary_plot(
    shap_values, X_test, feature_names=FEATURE_NAMES,
    plot_type="bar", class_names=TARGET_NAMES
)

## 6. SHAP — Explicación de una predicción individual

In [ ]:
# Seleccionar una muestra de ejemplo
sample_idx = 0
sample = X_test.iloc[[sample_idx]]
pred = model.predict(sample)[0]

print(f"Muestra: {sample.values[0]}")
print(f"Predicción: {TARGET_NAMES[pred]}")
print()

# Waterfall plot para la clase predicha
shap.initjs()
explanation = shap.Explanation(
    values=shap_values[pred][sample_idx],
    base_values=explainer.expected_value[pred],
    data=X_test.iloc[sample_idx].values,
    feature_names=FEATURE_NAMES,
)
shap.waterfall_plot(explanation)

## 7. SHAP — Dependence Plot

In [ ]:
# Dependencia de petal_length (típicamente la feature más importante)
shap.dependence_plot(
    "petal_length", shap_values[2], X_test,
    feature_names=FEATURE_NAMES, interaction_index="petal_width"
)

## 8. Conclusiones

- Completar tras ejecutar el notebook.
- Verificar que `petal_length` y `petal_width` dominan las decisiones.
- Analizar cómo las features contribuyen de forma diferente para cada clase.